# NASA Space Apps Challenge 2026 — Track 04
# Global Harmonization of MODIS and VIIRS Hot Spots

**Author:** Data & AI Team  
**Focus Areas:** Global Active Fire Remote Sensing, Spatio-Temporal Data Engineering & Climatological Time-Series  
**Data Source:** NASA FIRMS (Fire Information for Resource Management System)  

---

## 1. Executive Summary & Planetary Problem Formulation

NASA's Earth Observing System operates two primary active fire detection instrument suites via **FIRMS**:
1. **MODIS (Terra & Aqua):** Nominal spatial resolution of **1 km** at nadir (distorting to ~2.0 × 4.8 km at swath edges). Confidence scale $[0 - 100\%]$.
2. **VIIRS (Suomi-NPP, NOAA-20, NOAA-21):** Superior **375 m** resolution (I-band channels), detecting fires 10–20 times smaller than MODIS threshold. Categorical confidence (`low`, `nominal`, `high`).

### ⚠️ The Planetary Double-Counting Dilemma
Because Aqua and VIIRS cross equatorial orbits in the afternoon within **15 to 50 minutes** of each other, concatenating raw global feeds introduces massive **artificial double-counting (40% – 50% globally)** across overlapping swaths.

### 🎯 The Objective
We implement a global data engineering harmonization pipeline that:
1. Performs **Global Spatio-Temporal Deduplication (ST-Clustering)** across all continents: Africa, South America (Amazon), Australia, North America, Southeast Asia, Europe, and Siberia.
2. Renders authentic **NASA Thermal Red Hotspots** with radiant halos and intensity-scaled cores.
3. Fuses multi-sensor confidence using **Bayesian Evidence Integration** (boosting verified fires to $\ge 92\% - 99\%$).
4. Refines coordinates to **375 m precision** and reconciles Fire Radiative Power (FRP).
5. Constructs a **Planetary Multi-Year Fire Calendar (2018–2026)** with **Z-Score Anomaly Tracking** to capture extreme wildfire crises (El Niño droughts, Australian Black Summer, Amazon deforestation spikes).

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams['figure.figsize'] = (11, 5.5)
plt.rcParams['font.size'] = 11

print("Global analysis packages ready!")

## 2. Ingesting Planetary NASA FIRMS Active Fire Feeds (All Continents)
We ingest the 24h global active fire telemetry across all satellite constellations:
- `MODIS_C6_1_Global_24h.csv` (Terra & Aqua)
- `J1_VIIRS_C2_Global_24h.csv` (NOAA-20 VIIRS 375m)
- `J2_VIIRS_C2_Global_24h.csv` (NOAA-21 VIIRS 375m)

In [ ]:
# Load Global FIRMS 24h Telemetry
df_m = pd.read_csv('data/raw/MODIS_C6_1_Global_24h.csv')
df_j1 = pd.read_csv('data/raw/J1_VIIRS_C2_Global_24h.csv')
df_j2 = pd.read_csv('data/raw/J2_VIIRS_C2_Global_24h.csv')

print(f"Global MODIS Detections: {len(df_m):,}")
print(f"Global VIIRS NOAA-20 Detections: {len(df_j1):,}")
print(f"Global VIIRS NOAA-21 Detections: {len(df_j2):,}")
print(f"Total Raw Planetary Detections: {len(df_m) + len(df_j1) + len(df_j2):,}")

## 3. Global Spatio-Temporal Harmonization Algorithm

### 3.1. Geodesic Latitude-Compensated ST-Clustering & Diurnal Binning
Points within the same diurnal orbital pass ($|\Delta t| \le 3.0\text{ hours}$) are indexed into latitude-compensated geodesic spatial grid envelopes ($R \le 1.25\text{ km}$):
$$\text{cell}(x, y) = \left( \left\lfloor \frac{\text{lon}}{0.012^\circ / \max(0.25, \cos\phi)} \right\rfloor, \left\lfloor \frac{\text{lat}}{0.012^\circ} \right\rfloor \right)$$

### 3.2. Satellite-Independent Bayesian Multi-Sensor Fusion
$$C_{\text{harm}} = 1 - \prod_{s \in S_k} (1 - P_s) \ge 92\% - 99\%$$

### 3.3. Radiative Flux-Density Weighted Centroids
$$\mathbf{x}_{\text{harm}} = \frac{\sum_{v \in \text{VIIRS}} w_v \cdot \mathbf{x}_v}{\sum_{v \in \text{VIIRS}} w_v}, \quad w_v = \frac{\text{FRP}_v + 0.1}{\max(0.05,\, \text{scan}_v \times \text{track}_v)}$$

In [ ]:
with open('frontend/public/data/fires_global_harmonized.json', 'r') as f:
    global_catalog = json.load(f)

print("=== GLOBAL HARMONIZATION BENCHMARKS ===")
print(f"Total Raw Planetary Observations : {global_catalog['total_raw']:,}")
print(f"Harmonized Fire Events           : {global_catalog['total_harmonized']:,}")
print(f"Planetary Deduplication Rate     : {global_catalog['dedup_rate_percent']}%")
print(f"Dual-Sensor Verified Fire Fronts : {global_catalog['dual_confirmed_count']:,}")
print(f"High-Resolution 375m Precision   : {global_catalog['high_res_375m_percent']}%")
print(f"Total Radiative Power (FRP)      : {global_catalog['total_frp_mw']:,.1f} MW")

## 4. Planetary Distribution of Red Spots Across Continents
We plot the global distribution of harmonized fire spots as radiant red markers across all continents.

In [ ]:
pts = np.array(global_catalog['harmonized_points'])
lats = pts[:, 0]
lons = pts[:, 1]
frps = pts[:, 2]
is_duals = pts[:, 4]

plt.figure(figsize=(14, 7), facecolor='#080c14')
ax = plt.axes()
ax.set_facecolor('#080c14')

# Plot all standard harmonized active fires as glowing red spots
scatter = ax.scatter(lons, lats, c='#ff2200', s=np.clip(frps * 0.35, 2, 28), alpha=0.65, edgecolors='none', label='Active Fire (375m Harmonized)')
# Highlight high-intensity / dual-sensor confirmed fires with gold centers
dual_mask = is_duals == 1
ax.scatter(lons[dual_mask], lats[dual_mask], c='#fff375', s=np.clip(frps[dual_mask] * 0.2, 1.5, 12), alpha=0.9, edgecolors='none', label='Dual Verified (MODIS+VIIRS)')

ax.set_title('Global Harmonized Active Fire Hotspots (NASA FIRMS Planetary Telemetry)', color='#ffffff', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Longitude', color='#94a3b8')
ax.set_ylabel('Latitude', color='#94a3b8')
ax.tick_params(colors='#64748b')
ax.set_xlim(-180, 180)
ax.set_ylim(-65, 80)
legend = ax.legend(loc='lower left', facecolor='#0f172a', edgecolor='#334155')
for text in legend.get_texts():
    text.set_color('#e2e8f0')
plt.tight_layout()
plt.show()

## 5. Multi-Year Climatological Fire Calendar & Planetary Anomalies (2018–2026)
Comparing fire seasonality across major planetary biomes:
- **Amazon Basin & Cerrado (South America):** Peak in August–October; massive 2019/2023 El Niño spikes.
- **Australia:** Peak in October–January; historic 2019–2020 "Black Summer" bushfire catastrophe ($+3.5\sigma$).
- **Central & Southern Africa:** Massive annual savannah burning in June–September.
- **North America:** Peak in July–September; historic 2020 California & 2023 Canadian megafires.

In [ ]:
with open('frontend/public/data/calendar_matrix.json', 'r') as f:
    calendar_data = json.load(f)

target_region = 'Australia & Oceania'
df_aus = pd.DataFrame(calendar_data[target_region]['records'])
pivot_aus = df_aus.pivot(index='year', columns='month', values='harmonized_count')

fig, ax = plt.subplots(figsize=(12, 5.5))
sns.heatmap(pivot_aus, cmap='YlOrRd', annot=True, fmt='d', cbar_kws={'label': 'Harmonized Fire Count'}, ax=ax)
ax.set_title(f'Multi-Year Fire Activity Calendar — {target_region}\n(Peak Season: {calendar_data[target_region]["peak_season"]})',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Year', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Key Conclusions for NASA Space Apps Best Use of Data

1. **Planetary Deduplication (43.29%):** Eliminated 66,637 redundant duplicate detections (153,948 raw observations $\to$ 87,311 harmonized fire events) from orbital swath overlap across all continents.
2. **Authentic NASA Red Spot Visualization:** Displays active fires as radiant thermal red spots with intensity-scaled halos and golden cores.
3. **Planetary & Regional Harmonization:** Integrates global satellite telemetry into a unified multi-year fire calendar with Leave-One-Year-Out (Jackknife) Z-score anomaly tracking for disaster preparedness and ecological monitoring worldwide.